# Notebook 6 — Experiments 1 to 6: Baseline Models
**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

Five baseline models trained on TweetEval with FS1 features, then compared to
select the best performer.

| Exp | Model | Fills |
|---|---|---|
| 1 | Logistic Regression | Table 1 |
| 2 | Linear SVM | Table 1 |
| 3 | Multinomial Naive Bayes | Table 1 |
| 4 | Random Forest | Table 1 |
| 5 | XGBoost | Table 1 |
| 6 | Best model selection | Table 1 |

**Every experiment saves its predictions and probabilities to disk.**
Experiments 19, 20, 21 and 23 read those files instead of retraining. If this step
is skipped, four later experiments become expensive.

Tuning uses macro F1, not accuracy. A model predicting 'neutral' for everything
scores 45.9% accuracy on TweetEval while learning nothing.

## Cell 1: Setup

In [1]:
!pip install -q emoji xgboost scikit-learn pandas pyarrow

from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *

import pandas as pd, numpy as np, scipy.sparse as sp, pickle, time
from sklearn.model_selection import GridSearchCV

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 7.4 MB/s eta 0:00:00
Mounted at /content/drive
thesis_utils loaded.
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal


## Cell 2: Load Prepared Data
Everything comes from Notebook 5. No preprocessing happens here.

In [2]:
D = PATHS["data"]

tw_train = pd.read_parquet(D / "tw_train.parquet")
tw_test  = pd.read_parquet(D / "tw_test.parquet")

X_train = sp.load_npz(D / "tw_Xtrain_fs1.npz")
X_test  = sp.load_npz(D / "tw_Xtest_fs1.npz")

y_train = tw_train["sentiment"].values
y_test  = tw_test["sentiment"].values
sg_test = tw_test["subgroup_primary"].values

with open(D / "tw_vectorizer_fs1.pkl", "rb") as f:
    vectorizer = pickle.load(f)

print(f"Train : {X_train.shape[0]:,} rows x {X_train.shape[1]:,} features")
print(f"Test  : {X_test.shape[0]:,} rows")
print(f"\nClasses: {sorted(set(y_train))}")
print(f"Test subgroups:\n{pd.Series(sg_test).value_counts()}")

Train : 45,615 rows x 50,000 features
Test  : 12,284 rows

Classes: ['negative', 'neutral', 'positive']
Test subgroups:
formal         10398
other           1116
emoji-heavy      696
slang-heavy       60
sarcasm           14
Name: count, dtype: int64


## Cell 3: Shared Training Helper

One function used by every experiment so that tuning, evaluation and saving are
identical across models. The `save_predictions` call at the end is what makes the
later novelty experiments cheap.

In [3]:
def run_experiment(exp_id, model_name, estimator, param_grid,
                   X_tr=X_train, y_tr=y_train,
                   X_te=X_test,  y_te=y_test,
                   subgroups=sg_test, cv=5, dataset="TweetEval"):
    """
    Tune with GridSearchCV on macro F1, evaluate on the test set,
    save the model and — critically — save predictions and probabilities.
    """
    print("="*62)
    print(f"{exp_id.upper()} — {model_name}")
    print("="*62)

    t0 = time.time()
    gs = GridSearchCV(estimator, param_grid, cv=cv,
                      scoring="f1_macro",     # NOT accuracy
                      n_jobs=-1, verbose=0)
    gs.fit(X_tr, y_tr)
    elapsed = time.time() - t0

    best = gs.best_estimator_
    print(f"  best params : {gs.best_params_}")
    print(f"  CV macro F1 : {gs.best_score_:.4f}")
    print(f"  fit time    : {elapsed/60:.1f} min")

    y_pred  = best.predict(X_te)
    y_proba = best.predict_proba(X_te)

    res = evaluate_model(y_te, y_pred, y_proba, label=model_name)
    res["Dataset"]         = dataset
    res["Feature Used"]    = "TF-IDF unigram + bigram"
    res["Best Parameters"] = str(gs.best_params_)
    res["Fit Time (min)"]  = round(elapsed/60, 1)

    print(f"\n  Accuracy    : {res['Accuracy']:.4f}")
    print(f"  Macro F1    : {res['Macro F1']:.4f}")
    print(f"  Weighted F1 : {res['Weighted F1']:.4f}")
    print(f"  MCC         : {res['MCC']:.4f}")
    print(f"  Brier Score : {res['Brier Score']:.4f}")

    save_model(best, exp_id, model_name)
    save_predictions(exp_id, model_name, y_te, y_pred, y_proba,
                     subgroups, dataset=dataset)

    pred_df = load_predictions(exp_id, model_name, dataset)
    wga, worst_sg = worst_group_accuracy(pred_df)
    res["WGA"]                  = wga
    res["Worst Subgroup"]       = worst_sg
    res["Subgroup F1 Variance"] = subgroup_f1_variance(pred_df)

    print(f"  WGA         : {wga:.4f}  (worst: {worst_sg})")
    print()
    return res, best

results = []

## Cell 4: Experiment 1 — Logistic Regression

The most-cited baseline in the reviewed literature, which makes these numbers
directly comparable with published work. Also forms one half of the loss function
comparison in Experiment 19: LR minimises log loss, Linear SVM minimises hinge
loss, and both use identical features.

In [4]:
from sklearn.linear_model import LogisticRegression

res, model_lr = run_experiment(
    exp_id="exp01",
    model_name="LogisticRegression",
    estimator=LogisticRegression(max_iter=2000, random_state=SEED),
    param_grid={
        "C":       [0.01, 0.1, 1, 10],
        "penalty": ["l2"],
        "solver":  ["liblinear"],
    },
)
results.append(res)

EXP01 — LogisticRegression
  best params : {'C': 10, 'penalty': 'l2', 'solver': 'liblinear'}
  CV macro F1 : 0.6158
  fit time    : 0.7 min

  Accuracy    : 0.5800
  Macro F1    : 0.5672
  Weighted F1 : 0.5781
  MCC         : 0.3349
  Brier Score : 0.1860
  saved model -> exp01_LogisticRegression.pkl
  saved predictions -> exp01_LogisticRegression_TweetEval.parquet  (12,284 rows)
  WGA         : 0.5723  (worst: formal)



## Cell 5: Experiment 2 — Linear SVM

LinearSVC rather than an RBF kernel. RBF GridSearchCV on 45,615 training rows was
estimated at 6 to 12 hours per run, which is not supportable.

LinearSVC does not produce probabilities natively, so it is wrapped in
CalibratedClassifierCV. Probabilities are required for the AIF360 metrics and for
the HCER and MCE measures in Experiment 21.

In [5]:
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV

res, model_svm = run_experiment(
    exp_id="exp02",
    model_name="LinearSVM",
    estimator=CalibratedClassifierCV(
        LinearSVC(max_iter=3000, random_state=SEED, dual="auto"),
        cv=3, method="sigmoid"),
    param_grid={"estimator__C": [0.01, 0.1, 1, 10]},
)
results.append(res)

EXP02 — LinearSVM
  best params : {'estimator__C': 1}
  CV macro F1 : 0.6097
  fit time    : 2.2 min

  Accuracy    : 0.5876
  Macro F1    : 0.5648
  Weighted F1 : 0.5787
  MCC         : 0.3385
  Brier Score : 0.1766
  saved model -> exp02_LinearSVM.pkl
  saved predictions -> exp02_LinearSVM_TweetEval.parquet  (12,284 rows)
  WGA         : 0.5792  (worst: formal)



## Cell 6: Experiment 3 — Multinomial Naive Bayes

Fast probabilistic baseline. Also the reference point for Experiment 18, where
Complement Naive Bayes and SMOTE are compared against it on subgroup fairness.

TweetEval's negative class is only 19% of the data, so a standard MNB trained on
this distribution will underpredict negative sentiment. That is exactly what
Experiment 18 investigates.

In [6]:
from sklearn.naive_bayes import MultinomialNB

res, model_nb = run_experiment(
    exp_id="exp03",
    model_name="MultinomialNB",
    estimator=MultinomialNB(),
    param_grid={"alpha": [0.1, 0.5, 1.0, 2.0]},
)
results.append(res)

EXP03 — MultinomialNB
  best params : {'alpha': 0.1}
  CV macro F1 : 0.5846
  fit time    : 0.0 min

  Accuracy    : 0.5791
  Macro F1    : 0.5589
  Weighted F1 : 0.5722
  MCC         : 0.3219
  Brier Score : 0.1790
  saved model -> exp03_MultinomialNB.pkl
  saved predictions -> exp03_MultinomialNB_TweetEval.parquet  (12,284 rows)
  WGA         : 0.5748  (worst: formal)



## Cell 7: Experiment 4 — Random Forest

Bagging ensemble reference. Supports SHAP TreeExplainer, which matters for the
explainability stage in Experiment 17.

Note the reduced grid — Random Forest on 50,000 sparse features is slow, and a
larger grid would not change the conclusion.

In [7]:
from sklearn.ensemble import RandomForestClassifier

res, model_rf = run_experiment(
    exp_id="exp04",
    model_name="RandomForest",
    estimator=RandomForestClassifier(random_state=SEED, n_jobs=-1),
    param_grid={
        "n_estimators":     [200],
        "max_features":     ["sqrt"],
        "min_samples_leaf": [1, 5],
    },
)
results.append(res)

EXP04 — RandomForest
  best params : {'max_features': 'sqrt', 'min_samples_leaf': 1, 'n_estimators': 200}
  CV macro F1 : 0.4954
  fit time    : 56.0 min

  Accuracy    : 0.5394
  Macro F1    : 0.4169
  Weighted F1 : 0.4450
  MCC         : 0.2449
  Brier Score : 0.2002
  saved model -> exp04_RandomForest.pkl
  saved predictions -> exp04_RandomForest_TweetEval.parquet  (12,284 rows)
  WGA         : 0.5263  (worst: formal)



## Cell 8: Experiment 5 — XGBoost

Level-wise tree growth. Compared against LightGBM's leaf-wise growth in
Experiment 19, where both use identical features so any fairness difference is
attributable to growth strategy alone.

XGBoost needs integer labels, so they are encoded and decoded around the fit.

In [8]:
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder().fit(y_train)
y_train_enc = le.transform(y_train)
y_test_enc  = le.transform(y_test)

print(f"Label encoding: {dict(zip(le.classes_, le.transform(le.classes_)))}\n")

gs = GridSearchCV(
    XGBClassifier(random_state=SEED, n_jobs=-1,
                  eval_metric="mlogloss", tree_method="hist"),
    {"n_estimators": [200], "learning_rate": [0.1], "max_depth": [5, 7]},
    cv=5, scoring="f1_macro", n_jobs=-1)

t0 = time.time(); gs.fit(X_train, y_train_enc); elapsed = time.time() - t0

best_xgb = gs.best_estimator_
y_pred   = le.inverse_transform(best_xgb.predict(X_test))
y_proba  = best_xgb.predict_proba(X_test)

res = evaluate_model(y_test, y_pred, y_proba, label="XGBoost")
res.update({"Dataset": "TweetEval", "Feature Used": "TF-IDF unigram + bigram",
            "Best Parameters": str(gs.best_params_),
            "Fit Time (min)": round(elapsed/60, 1)})

save_model(best_xgb, "exp05", "XGBoost")
save_predictions("exp05", "XGBoost", y_test, y_pred, y_proba, sg_test)

pred_df = load_predictions("exp05", "XGBoost")
wga, worst = worst_group_accuracy(pred_df)
res["WGA"] = wga; res["Worst Subgroup"] = worst
res["Subgroup F1 Variance"] = subgroup_f1_variance(pred_df)

print(f"Macro F1 : {res['Macro F1']:.4f}")
print(f"WGA      : {wga:.4f}  (worst: {worst})")
results.append(res)

# label encoder is needed again in later notebooks
with open(PATHS["data"] / "label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

Label encoding: {'negative': np.int64(0), 'neutral': np.int64(1), 'positive': np.int64(2)}

  saved model -> exp05_XGBoost.pkl
  saved predictions -> exp05_XGBoost_TweetEval.parquet  (12,284 rows)
Macro F1 : 0.4519
WGA      : 0.5201  (worst: emoji-heavy)


## Cell 9: Experiment 6 — Best Baseline Selection

Ranks all five models by macro F1 and produces Table 1.

Worst-Group Accuracy is reported alongside the headline metrics deliberately. A
model can lead on macro F1 while failing badly on one linguistic subgroup, and
selecting on average performance alone would hide that.

In [9]:
table1 = pd.DataFrame(results)

cols = ["Model", "Dataset", "Feature Used", "Accuracy", "Precision", "Recall",
        "Macro F1", "Weighted F1", "MCC", "Cohen Kappa", "Brier Score",
        "WGA", "Worst Subgroup", "Subgroup F1 Variance",
        "Best Parameters", "Fit Time (min)"]
table1 = table1[[c for c in cols if c in table1.columns]]
table1 = table1.sort_values("Macro F1", ascending=False).reset_index(drop=True)

table1["Best / Not Best"] = ["BEST"] + ["Not Best"] * (len(table1) - 1)

print("="*80)
print("TABLE 1 — BASELINE MODEL PERFORMANCE")
print("="*80)
print(table1[["Model", "Accuracy", "Macro F1", "Weighted F1",
              "WGA", "Worst Subgroup", "Best / Not Best"]].to_string(index=False))

best_model_name = table1.iloc[0]["Model"]
print(f"\nBest baseline by macro F1: {best_model_name}")
print(f"But note the WGA column — check whether the best model on macro F1")
print(f"is also the most consistent across subgroups.")

save_result_table(table1, "Table1_Baseline_Performance")

with open(PATHS["data"] / "best_baseline.json", "w") as f:
    json.dump({"best_model": best_model_name,
               "macro_f1": float(table1.iloc[0]["Macro F1"])}, f)

TABLE 1 — BASELINE MODEL PERFORMANCE
             Model  Accuracy  Macro F1  Weighted F1      WGA Worst Subgroup Best / Not Best
LogisticRegression  0.580023  0.567213     0.578076 0.572322         formal            BEST
         LinearSVM  0.587594  0.564832     0.578698 0.579246         formal        Not Best
     MultinomialNB  0.579127  0.558945     0.572164 0.574822         formal        Not Best
           XGBoost  0.554217  0.451909     0.483069 0.520115    emoji-heavy        Not Best
      RandomForest  0.539401  0.416909     0.445017 0.526255         formal        Not Best

Best baseline by macro F1: LogisticRegression
But note the WGA column — check whether the best model on macro F1
is also the most consistent across subgroups.
  saved table -> Table1_Baseline_Performance.csv


## Cell 10: Subgroup Breakdown for Every Baseline

Not part of the original Experiment 6 output, but required for Experiment 19.
Only one model proceeds past Experiment 6, so if subgroup fairness is not computed
for all five models here, the loss function comparison in Experiment 19 cannot be
made at all.

In [10]:
model_ids = [("exp01", "LogisticRegression"), ("exp02", "LinearSVM"),
             ("exp03", "MultinomialNB"),      ("exp04", "RandomForest"),
             ("exp05", "XGBoost")]

rows = []
for exp_id, name in model_ids:
    pred_df = load_predictions(exp_id, name)
    rep = subgroup_report(pred_df)
    rep.insert(0, "Model", name)
    rows.append(rep)

subgroup_all = pd.concat(rows, ignore_index=True)

print("SUBGROUP MACRO F1 BY MODEL")
print("="*70)
pivot = subgroup_all.pivot(index="Model", columns="Subgroup", values="Macro F1")
print(pivot.round(4).to_string())

save_result_table(subgroup_all, "Table1b_Baseline_Subgroup_Breakdown")

print("\nThis table is the input to Experiment 19 (Novelty 2).")
print("Compare the LogisticRegression and LinearSVM rows: identical features,")
print("different loss function. Any gap between them is attributable to that.")

SUBGROUP MACRO F1 BY MODEL
Subgroup            emoji-heavy  formal   other  sarcasm  slang-heavy
Model                                                                
LinearSVM                0.5711  0.5553  0.5871   0.6465       0.6959
LogisticRegression       0.5815  0.5575  0.5777   0.6465       0.6824
MultinomialNB            0.5615  0.5517  0.5464   0.5460       0.5765
RandomForest             0.4344  0.4083  0.4749   0.2456       0.4868
XGBoost                  0.4375  0.4505  0.4411   0.2222       0.4001
  saved table -> Table1b_Baseline_Subgroup_Breakdown.csv

This table is the input to Experiment 19 (Novelty 2).
Compare the LogisticRegression and LinearSVM rows: identical features,
different loss function. Any gap between them is attributable to that.


## Cell 11: Done

In [11]:
print("="*62)
print("NOTEBOOK 6 COMPLETE — EXPERIMENTS 1 TO 6")
print("="*62)
print("\nSaved predictions:")
list_predictions()
print("\nSaved tables: Table1_Baseline_Performance, Table1b_Baseline_Subgroup_Breakdown")
print("\nNext: Notebook 7 — Experiments 7 to 10 (features and tuning)")

NOTEBOOK 6 COMPLETE — EXPERIMENTS 1 TO 6

Saved predictions:
  exp01_LogisticRegression_TweetEval.parquet
  exp02_LinearSVM_TweetEval.parquet
  exp03_MultinomialNB_TweetEval.parquet
  exp04_RandomForest_TweetEval.parquet
  exp05_XGBoost_TweetEval.parquet

Saved tables: Table1_Baseline_Performance, Table1b_Baseline_Subgroup_Breakdown

Next: Notebook 7 — Experiments 7 to 10 (features and tuning)
